In [23]:
import os
import json
import mne
import numpy as np
import pandas as pd
from scipy.signal import resample
import warnings
warnings.filterwarnings("ignore")
from collections import defaultdict
import scipy.io as sio
import mne

In [25]:
# multi-scale sampling rates
SAMPLE_RATE_LIST = [200, 100, 50]  # Hz

# fixed segment length in samples (timestamps)
SEG_LEN = 200  # number of time steps per segment
# overlap in samples (timestamps), not in seconds
OVERLAP = 100    # e.g. 100 means 50% overlap for SEG_LEN=200

assert 0 <= OVERLAP < SEG_LEN, "OVERLAP must be in [0, SEG_LEN)."

sub_folder_path = f"L{SEG_LEN}"
sub_folder_path

'L200'

## Official script

In [17]:
filenames = []
for filename in os.listdir("APAVA/"):
  filenames.append(filename)

In [18]:
# filenames.sort()
filenames

['preproctrials01.mat',
 'preproctrials02.mat',
 'preproctrials03.mat',
 'preproctrials04.mat',
 'preproctrials05.mat',
 'preproctrials06.mat',
 'preproctrials07.mat',
 'preproctrials08.mat',
 'preproctrials09.mat',
 'preproctrials10.mat',
 'preproctrials11.mat',
 'preproctrials12.mat',
 'preproctrials13.mat',
 'preproctrials14.mat',
 'preproctrials15.mat',
 'preproctrials16.mat',
 'preproctrials17.mat',
 'preproctrials18.mat',
 'preproctrials19.mat',
 'preproctrials20.mat',
 'preproctrials21.mat',
 'preproctrials22.mat',
 'preproctrials23.mat']

In [21]:
AD_positive = [1,3,6,8,9,11,12,13,15,17,19,21]
labels = np.empty((23, 2))
for i in range(len(labels)):
  # The first one is AD label (0 for healthy; 1 for AD patient)
  # The second one is the subject label (the order of subject, ranging from 1 to 23.
  labels[i][1] = i + 1
  if i+1 in AD_positive:
    labels[i][0] = 1
  else:
    labels[i][0] = 0
labels = labels.astype(int)
labels

array([[ 1,  1],
       [ 0,  2],
       [ 1,  3],
       [ 0,  4],
       [ 0,  5],
       [ 1,  6],
       [ 0,  7],
       [ 1,  8],
       [ 1,  9],
       [ 0, 10],
       [ 1, 11],
       [ 1, 12],
       [ 1, 13],
       [ 0, 14],
       [ 1, 15],
       [ 0, 16],
       [ 1, 17],
       [ 0, 18],
       [ 1, 19],
       [ 0, 20],
       [ 1, 21],
       [ 0, 22],
       [ 0, 23]])

In [19]:
def resample_time_series(data, original_fs, target_fs):
    T, C = data.shape
    new_length = int(T * target_fs / original_fs)
    
    resampled_data = np.zeros((new_length, C))
    for i in range(C):
        resampled_data[:, i] = resample(data[:, i], new_length)
        
    return resampled_data


def compute_step(seg_len, overlap):
    """
    Compute sliding step (in samples) given segment length and overlap.
    """
    step = seg_len - overlap
    if step <= 0:
        raise ValueError(f"Invalid overlap={overlap}: step <= 0.")
    return step


def compute_num_segments(num_samples, seg_len, step):
    """
    Compute how many segments can be extracted from a sequence
    with given segment length and step.
    """
    if num_samples < seg_len:
        return 0
    return 1 + (num_samples - seg_len) // step

## PASS 1: Find total number of segments across all subjects

In [27]:
subject_segment_counts = defaultdict(lambda: defaultdict(int))
N_total = 0
# the order of input channels are retrieved from the .mat files. We hard code them here for consistency.
input_channels = ['C3', 'C4', 'F3', 'F4', 'F7', 'F8', 'Fp1', 'Fp2', 'O1', 'O2', 'P3', 'P4', 'T3', 'T4', 'T5', 'T6']
n_channels = len(input_channels)

# step (timestamps)
STEP = compute_step(SEG_LEN, OVERLAP)
print("SEG_LEN =", SEG_LEN, "OVERLAP =", OVERLAP, "STEP =", STEP)
print("===================================")


sub = 1
for i in range(len(filenames)):
    # print('Dataset/'+filename)
    path = "APAVA/" + filenames[i]
    mat = sio.loadmat(path)
    mat_np = mat['data']

    # Get epoch number for each subject
    epoch_num = len(mat_np[0,0][2][0])
    print("Epoch number: ",epoch_num)
    # Each epoch has shape (1280, 16)
    raw_shape = np.zeros((epoch_num, 1280, 16)).shape
    features = []
    # Store in temp
    s_n_seg = 0
    for j in range(epoch_num):
        temp = np.transpose(mat_np[0,0][2][0][j])
        T_raw = temp.shape[0]
        original_fs = 256
        for fs in SAMPLE_RATE_LIST:
            # resampled length if each trial
            T_res = int(T_raw * fs / original_fs)
            # compute number of segments
            n_seg = compute_num_segments(T_res, SEG_LEN, STEP)
            subject_segment_counts[sub][fs] += n_seg
            s_n_seg += n_seg
            N_total += n_seg
    for fs in SAMPLE_RATE_LIST:
        # total segments of each sampling rate for this subject
        print(f"Subject {sub}: Sampling Rate {fs} Hz: Segments = {subject_segment_counts[sub][fs]}")
    print(f"Subject {sub}: Total segments = {s_n_seg}")
    sub += 1
    print("-------------------------------------\n")

print("\n===================================")
print("Total segments N_total =", N_total)
print("Channels =", n_channels)
print("===================================")

if N_total == 0:
    raise RuntimeError("No segments found. Please check SEG_LEN / OVERLAP.")

SEG_LEN = 200 OVERLAP = 100 STEP = 100
Epoch number:  35
Subject 1: Sampling Rate 200 Hz: Segments = 315
Subject 1: Sampling Rate 100 Hz: Segments = 140
Subject 1: Sampling Rate 50 Hz: Segments = 35
Subject 1: Total segments = 490
-------------------------------------

Epoch number:  25
Subject 2: Sampling Rate 200 Hz: Segments = 225
Subject 2: Sampling Rate 100 Hz: Segments = 100
Subject 2: Sampling Rate 50 Hz: Segments = 25
Subject 2: Total segments = 350
-------------------------------------

Epoch number:  10
Subject 3: Sampling Rate 200 Hz: Segments = 90
Subject 3: Sampling Rate 100 Hz: Segments = 40
Subject 3: Sampling Rate 50 Hz: Segments = 10
Subject 3: Total segments = 140
-------------------------------------

Epoch number:  33
Subject 4: Sampling Rate 200 Hz: Segments = 297
Subject 4: Sampling Rate 100 Hz: Segments = 132
Subject 4: Sampling Rate 50 Hz: Segments = 33
Subject 4: Total segments = 462
-------------------------------------

Epoch number:  1
Subject 5: Sampling Ra

In [33]:
output_root = os.path.join('Processed', sub_folder_path, 'APAVA')
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 3))

X path: Processed\L200\APAVA\X.dat
y path: Processed\L200\APAVA\y.dat


## PASS 2: Process and store segments into memmap

In [34]:
cur = 0  # current index in memmap
total_seconds_all = 0  # used for total hours calculation
sub = 1
for i in range(len(filenames)):
    # print('Dataset/'+filename)
    path = "APAVA/" + filenames[i]
    mat = sio.loadmat(path)
    mat_np = mat['data']
    label, subject_id = labels[i]

    # Get epoch number for each subject
    epoch_num = len(mat_np[0,0][2][0])
    print("Subject ID: ", subject_id)
    print("Epoch number in subject: ",epoch_num)
    # Each epoch has shape (1280, 16)
    raw_shape = np.zeros((epoch_num, 1280, 16)).shape
    features = []
    # Store in temp
    s_n_seg = 0
    for j in range(epoch_num):
        temp = np.transpose(mat_np[0,0][2][0][j])
        T_raw = temp.shape[0]
        original_fs = 256
        total_seconds_all += temp.shape[0] / original_fs
        for fs in SAMPLE_RATE_LIST:
            # resample to target fs
            data = resample_time_series(temp, original_fs, fs)
            T_res, _ = data.shape
            # print(f"fs={fs}Hz: resampled shape={data.shape}")

            # overlapped sliding window with fixed STEP (in timestamps)
            starts = np.arange(0, T_res - SEG_LEN + 1, STEP, dtype=int)
            # print(f"fs={fs}Hz: segments={len(starts)}")

            for s in starts:
                if cur >= N_total:
                    raise RuntimeError("Exceeded predicted N_total.")

                X_mm[cur] = data[s:s + SEG_LEN]   # (SEG_LEN, C)
                y_mm[cur, 0] = float(label)       # label
                y_mm[cur, 1] = float(subject_id)         # subject ID
                y_mm[cur, 2] = float(fs)          # sample rate (scale)
                cur += 1
    for fs in SAMPLE_RATE_LIST:
        # total segments of each sampling rate for this subject
        print(f"Subject {sub}: Sampling Rate {fs} Hz: Segments = {subject_segment_counts[sub][fs]}")
    print(f"Subject {sub}: Total segments = {s_n_seg}")
    sub += 1
    print("-------------------------------------\n")

total_hours_all = total_seconds_all / 3600.0
print("DONE: cur =", cur, " expected N_total =", N_total)
print(f"Total hours across all subjects: {total_hours_all:.2f} hours")

# flush
del X_mm
del y_mm

# save meta
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE_LIST": SAMPLE_RATE_LIST,
    "OVERLAP": int(OVERLAP),   # in samples
    "STEP": int(STEP),
    "X_path": X_path,
    "y_path": y_path,
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

Subject ID:  1
Epoch number in subject:  35
Subject 1: Sampling Rate 200 Hz: Segments = 315
Subject 1: Sampling Rate 100 Hz: Segments = 140
Subject 1: Sampling Rate 50 Hz: Segments = 35
Subject 1: Total segments = 0
-------------------------------------

Subject ID:  2
Epoch number in subject:  25
Subject 2: Sampling Rate 200 Hz: Segments = 225
Subject 2: Sampling Rate 100 Hz: Segments = 100
Subject 2: Sampling Rate 50 Hz: Segments = 25
Subject 2: Total segments = 0
-------------------------------------

Subject ID:  3
Epoch number in subject:  10
Subject 3: Sampling Rate 200 Hz: Segments = 90
Subject 3: Sampling Rate 100 Hz: Segments = 40
Subject 3: Sampling Rate 50 Hz: Segments = 10
Subject 3: Total segments = 0
-------------------------------------

Subject ID:  4
Epoch number in subject:  33
Subject 4: Sampling Rate 200 Hz: Segments = 297
Subject 4: Sampling Rate 100 Hz: Segments = 132
Subject 4: Sampling Rate 50 Hz: Segments = 33
Subject 4: Total segments = 0
---------------------

## Load and check the processed data

In [35]:
import json
import numpy as np

# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]
X_path = meta["X_path"]
y_path = meta["y_path"]

print("Meta:")
print("  N =", N)
print("  T =", T)
print("  C =", C)
print("  X_path =", X_path)
print("  y_path =", y_path)
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 3))

# subject_id is stored in y[:, 1]
subject_ids = np.unique(y_mm[:, 1]).astype(int)

for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 1] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 3)       # logical y shape for this subject

    # get sampling rates for all segments of this subject
    fs_subject = y_mm[idx, 2]  # shape (n_seg,)


    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

# option: delete memmap views
del X_mm, y_mm

Meta:
  N = 9282
  T = 200
  C = 16
  X_path = Processed\L200\APAVA\X.dat
  y_path = Processed\L200\APAVA\y.dat
-------------------------------------
Subject ID 001: X shape=(490, 200, 16), y shape=(490, 3)
Subject ID 002: X shape=(350, 200, 16), y shape=(350, 3)
Subject ID 003: X shape=(140, 200, 16), y shape=(140, 3)
Subject ID 004: X shape=(462, 200, 16), y shape=(462, 3)
Subject ID 005: X shape=(14, 200, 16), y shape=(14, 3)
Subject ID 006: X shape=(308, 200, 16), y shape=(308, 3)
Subject ID 007: X shape=(42, 200, 16), y shape=(42, 3)
Subject ID 008: X shape=(448, 200, 16), y shape=(448, 3)
Subject ID 009: X shape=(252, 200, 16), y shape=(252, 3)
Subject ID 010: X shape=(532, 200, 16), y shape=(532, 3)
Subject ID 011: X shape=(658, 200, 16), y shape=(658, 3)
Subject ID 012: X shape=(518, 200, 16), y shape=(518, 3)
Subject ID 013: X shape=(406, 200, 16), y shape=(406, 3)
Subject ID 014: X shape=(546, 200, 16), y shape=(546, 3)
Subject ID 015: X shape=(644, 200, 16), y shape=(644, 3)